# RCAEval Exploratory Data Analysis (EDA)

**Project:** Agentic Incident-Response Copilot (CMPE 295)  
**Dataset:** public [RCAEval](https://github.com/phamquiluan/RCAEval) benchmark (not created by this team)  
**Notebook:** `EDA/RCAEval_EDA.ipynb`  
**Summary:** `EDA/RCAEval_EDA_SUMMARY.md`

This notebook explains the RCAEval layout, inspects columns/telemetry, and summarizes all 9 suites (RE1/RE2/RE3 × Online Boutique / Sock Shop / Train Ticket).


## 1. Setup and locate the dataset

Set `RCAEVAL_DIR` if the data is not under `../datasets/RCAEval` relative to the repo root.


In [3]:
from __future__ import annotations

import os
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "EDA").is_dir() and (REPO_ROOT / "eval").is_dir():
    pass
elif (REPO_ROOT.parent / "eval").is_dir():
    REPO_ROOT = REPO_ROOT.parent

CANDIDATE_ROOTS = [
    Path(os.environ["RCAEVAL_DIR"]) if os.environ.get("RCAEVAL_DIR") else None,
    REPO_ROOT.parent / "datasets" / "RCAEval",
    REPO_ROOT / "datasets" / "RCAEval",
    Path.home() / "Desktop" / "Masters RCA" / "datasets" / "RCAEval",
]

def find_data_root() -> Path:
    for root in CANDIDATE_ROOTS:
        if root and root.is_dir() and any(root.glob("re*_*")):
            return root
    raise FileNotFoundError(
        "RCAEval data not found. Download cases and set RCAEVAL_DIR, "
        "or place them under ../datasets/RCAEval"
    )

DATA_ROOT = find_data_root()
print("Repo:", REPO_ROOT)
print("RCAEval data root:", DATA_ROOT)
print("Example case folders:", sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir())[:5])


Repo: /Users/charan/Desktop/Masters RCA/repo
RCAEval data root: /Users/charan/Desktop/Masters RCA/datasets/RCAEval
Example case folders: ['.cache', 're1ob_adservice_cpu_1', 're1ob_adservice_cpu_2', 're1ob_adservice_cpu_3', 're1ob_adservice_cpu_4']


## 2. What is RCAEval? (apps × releases)

- **Apps (systems under test)**
  - `ob` = Online Boutique
  - `ss` = Sock Shop
  - `tt` = Train Ticket
- **Releases (dataset editions)**
  - `re1` = mostly metrics-only
  - `re2` = richer multi-source (metrics + logs/traces when available)
  - `re3` = functional/code-style faults (`f1`…`f5`)

A **suite** is one pair, e.g. `re2ob` = Release 2 + Online Boutique.  
Each case folder name encodes the label: `{suite}_{service}_{fault}_{rep}`.


In [5]:
CASE_RE = re.compile(
    r'^(re[123](?:ob|ss|tt))_(.+)_(cpu|mem|disk|delay|loss|socket|f[1-5])_(\d+)$'
)
ALL_SUITES = [
    "re1ob", "re1ss", "re1tt",
    "re2ob", "re2ss", "re2tt",
    "re3ob", "re3ss", "re3tt",
]

def parse_case(path: Path):
    m = CASE_RE.match(path.name)
    if not m:
        return None
    return {
        "path": path,
        "name": path.name,
        "suite": m.group(1),
        "service": m.group(2),
        "fault": m.group(3),
        "rep": m.group(4),
    }

def list_cases(suite: str):
    out = []
    for p in sorted(DATA_ROOT.iterdir()):
        if not p.is_dir():
            continue
        c = parse_case(p)
        if c and c["suite"] == suite:
            out.append(c)
    return out

inventory = []
for suite in ALL_SUITES:
    cases = list_cases(suite)
    faults = Counter(c["fault"] for c in cases)
    services = Counter(c["service"] for c in cases)
    inventory.append({
        "suite": suite,
        "n_cases": len(cases),
        "n_fault_types": len(faults),
        "faults": dict(sorted(faults.items())),
        "n_services": len(services),
        "services": sorted(services),
    })

inv_df = pd.DataFrame(inventory)
print("Total cases:", inv_df["n_cases"].sum())
inv_df[["suite", "n_cases", "n_fault_types", "n_services", "faults"]]


Total cases: 735


## 3. One case walk-through (columns / files)

We open a single `re2ob` case and explain the tables:

- `metrics.parquet` — time series columns (usually `{service}_{signal}`)
- `logs.parquet` — container log lines (RE2/RE3 when present)
- `traces.parquet` — distributed traces (when present)
- `inject_time.txt` — second when the fault starts


In [7]:
METRIC_SUFFIXES = [
    "_latency-90", "_latency-50", "_diskio", "_workload", "_socket",
    "_error", "_cpu", "_mem",
]

def metric_to_service(col: str):
    if col == "time":
        return None
    for suf in METRIC_SUFFIXES:
        if col.endswith(suf):
            svc = col[: -len(suf)]
            return "redis-cart" if svc == "redis" else svc
    return None

re2ob_cases = list_cases("re2ob")
assert re2ob_cases, "re2ob cases missing — check DATA_ROOT"
example = re2ob_cases[0]
print("Example case:", example["name"])
print("True service (label):", example["service"])
print("Fault type:", example["fault"])

files = sorted(p.name for p in example["path"].iterdir())
print("Files:", files)

inject = int((example["path"] / "inject_time.txt").read_text().strip())
metrics = pd.read_parquet(example["path"] / "metrics.parquet")
print("inject_time:", inject)
print("metrics shape:", metrics.shape)
print("metrics columns (first 15):", list(metrics.columns)[:15])

svc_map = {c: metric_to_service(c) for c in metrics.columns if c != "time"}
print("Unique services in metric columns:", sorted({v for v in svc_map.values() if v})[:20])
metrics.head(3)


Example case: re2ob_checkoutservice_cpu_1
True service (label): checkoutservice
Fault type: cpu
Files: ['inject_time.txt', 'logs.parquet', 'metrics.parquet', 'traces.parquet']
inject_time: 1705354566
metrics shape: (1441, 73)
metrics columns (first 15): ['time', 'adservice_cpu', 'cartservice_cpu', 'checkoutservice_cpu', 'currencyservice_cpu', 'emailservice_cpu', 'frontend_cpu', 'paymentservice_cpu', 'productcatalogservice_cpu', 'recommendationservice_cpu', 'redis_cpu', 'shippingservice_cpu', 'adservice_mem', 'cartservice_mem', 'checkoutservice_mem']
Unique services in metric columns: ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice', 'frontend', 'frontend-external', 'paymentservice', 'productcatalogservice', 'recommendationservice', 'redis-cart', 'shippingservice']


In [8]:
# Logs / traces if present
logs_path = example["path"] / "logs.parquet"
traces_path = example["path"] / "traces.parquet"

if logs_path.exists():
    logs = pd.read_parquet(logs_path)
    print("logs shape:", logs.shape)
    print("logs columns:", list(logs.columns))
    print(logs.head(2))
else:
    print("No logs.parquet in this case")

if traces_path.exists():
    traces = pd.read_parquet(traces_path)
    print("traces shape:", traces.shape)
    print("traces columns:", list(traces.columns))
    print(traces.head(2))
else:
    print("No traces.parquet in this case")


logs shape: (171322, 3)
logs columns: ['timestamp', 'container_name', 'message']
    timestamp   container_name                        message
0  1705353846  currencyservice  conversion request successful
1  1705353846  currencyservice  conversion request successful
traces shape: (391997, 11)
traces columns: ['time', 'traceID', 'spanID', 'serviceName', 'methodName', 'operationName', 'parentSpanID', 'startTimeMillis', 'startTime', 'duration', 'statusCode']
    time                           traceID            spanID      serviceName methodName  \
0  21:24  46d5d1e1fc25a09db4a81d4d105cee3d  de79080386c4c5c3  currencyservice    Convert   
1  21:24  46d5d1e1fc25a09db4a81d4d105cee3d  433c41ba31b519dd  frontendservice    Convert   

                              operationName      parentSpanID  startTimeMillis         startTime  duration  statusCode  
0  grpc.hipstershop.CurrencyService/Convert  433c41ba31b519dd    1705353846065  1705353846065999       186           0  
1       hipstershop.C

## 4. Deep EDA helpers

For each case we measure:

- metric width (# columns), missing rate, constant columns, sampling interval
- pre vs post `inject_time` row counts
- log/trace volume when available
- label leakage note (folder name encodes truth)


In [10]:
def missing_rate(df: pd.DataFrame) -> float:
    if df.empty:
        return 0.0
    return float(df.isna().mean().mean())

def constant_cols(df: pd.DataFrame) -> int:
    n = 0
    for c in df.columns:
        if c == "time":
            continue
        if df[c].nunique(dropna=True) <= 1:
            n += 1
    return n

def sampling_interval(times: pd.Series):
    if len(times) < 3:
        return None
    diffs = np.diff(sorted(times.dropna().astype(float).tolist()))
    diffs = diffs[diffs > 0]
    return float(np.median(diffs)) if len(diffs) else None

def parquet_num_rows(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        import pyarrow.parquet as pq
        return int(pq.ParquetFile(path).metadata.num_rows)
    except Exception:
        return int(len(pd.read_parquet(path)))

def analyze_case(case: dict) -> dict:
    metrics = pd.read_parquet(case["path"] / "metrics.parquet")
    t0 = int((case["path"] / "inject_time.txt").read_text().strip())
    metric_cols = [c for c in metrics.columns if c != "time"]
    pre = metrics[metrics["time"] < t0]
    post = metrics[metrics["time"] >= t0]
    logs_rows = parquet_num_rows(case["path"] / "logs.parquet")
    traces_rows = parquet_num_rows(case["path"] / "traces.parquet")
    services = sorted({metric_to_service(c) for c in metric_cols if metric_to_service(c)})
    return {
        "case": case["name"],
        "suite": case["suite"],
        "service": case["service"],
        "fault": case["fault"],
        "inject_time": t0,
        "metrics_rows": len(metrics),
        "metrics_cols": len(metric_cols),
        "metric_missing_rate": missing_rate(metrics[metric_cols]) if metric_cols else 0.0,
        "constant_metric_cols": constant_cols(metrics),
        "sampling_interval_sec": sampling_interval(metrics["time"]),
        "pre_rows": len(pre),
        "post_rows": len(post),
        "logs_rows": logs_rows,
        "traces_rows": traces_rows,
        "n_services_in_metrics": len(services),
        "leakage_folder_encodes_label": True,
    }

print("Helpers ready.")


Helpers ready.


## 5. Suite-level EDA (all 9 suites)

This can take a few minutes because RE2/RE3 logs/traces are large.  
Set `QUICK = True` to sample a few cases per suite for a fast classroom demo.


In [12]:
QUICK = False  # True = first 6 cases/suite for a fast pass
MAX_PER_SUITE = 6 if QUICK else None

all_rows = []
suite_summaries = []

for suite in ALL_SUITES:
    cases = list_cases(suite)
    if not cases:
        print(f"{suite}: missing")
        continue
    use = cases[:MAX_PER_SUITE] if MAX_PER_SUITE else cases
    print(f"Analyzing {suite}: {len(use)}/{len(cases)} cases ...")
    rows = [analyze_case(c) for c in use]
    all_rows.extend(rows)
    df = pd.DataFrame(rows)
    suite_summaries.append({
        "suite": suite,
        "n_cases_analyzed": len(df),
        "n_cases_total": len(cases),
        "avg_metrics_cols": round(float(df["metrics_cols"].mean()), 2),
        "avg_sampling_interval_sec": round(float(df["sampling_interval_sec"].dropna().mean()), 3)
            if df["sampling_interval_sec"].notna().any() else None,
        "avg_metric_missing_rate": round(float(df["metric_missing_rate"].mean()), 5),
        "avg_constant_metric_cols": round(float(df["constant_metric_cols"].mean()), 2),
        "avg_logs_rows": round(float(df["logs_rows"].mean()), 1),
        "avg_traces_rows": round(float(df["traces_rows"].mean()), 1),
        "fault_counts": dict(Counter(df["fault"])),
    })

summary_df = pd.DataFrame(suite_summaries)
cases_df = pd.DataFrame(all_rows)
summary_df


Analyzing re1ob: 125/125 cases ...
Analyzing re1ss: 125/125 cases ...
Analyzing re1tt: 125/125 cases ...
Analyzing re2ob: 90/90 cases ...
Analyzing re2ss: 90/90 cases ...
Analyzing re2tt: 90/90 cases ...
Analyzing re3ob: 30/30 cases ...
Analyzing re3ss: 30/30 cases ...
Analyzing re3tt: 30/30 cases ...


In [13]:
# Save notebook outputs next to this file for the PR summary package
out_dir = REPO_ROOT / "EDA" / "artifacts"
out_dir.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(out_dir / "suite_eda_summary.csv", index=False)
cases_df.to_csv(out_dir / "case_eda_rows.csv", index=False)
print("Wrote:", out_dir / "suite_eda_summary.csv")
print("Wrote:", out_dir / "case_eda_rows.csv")
summary_df[["suite", "n_cases_total", "avg_metrics_cols", "avg_logs_rows", "avg_traces_rows", "avg_metric_missing_rate"]]


Wrote: /Users/charan/Desktop/Masters RCA/repo/EDA/artifacts/suite_eda_summary.csv
Wrote: /Users/charan/Desktop/Masters RCA/repo/EDA/artifacts/case_eda_rows.csv


## 6. Focus: RE2-OB column and fault profile

RE2 Online Boutique is our primary reference suite for metric validation (90 labelled cases).


In [15]:
re2 = cases_df[cases_df["suite"] == "re2ob"].copy()
if re2.empty:
    # analyze fully if quick mode skipped it partially
    re2 = pd.DataFrame([analyze_case(c) for c in list_cases("re2ob")])

print("RE2-OB cases:", len(re2))
print("Fault counts:\n", re2["fault"].value_counts().sort_index())
print("Service counts:\n", re2["service"].value_counts().sort_index())
print(
    "Telemetry averages — metrics_cols:"
    f" {re2['metrics_cols'].mean():.1f}, logs: {re2['logs_rows'].mean():.0f},"
    f" traces: {re2['traces_rows'].mean():.0f}, missing: {re2['metric_missing_rate'].mean():.4f},"
    f" constant cols: {re2['constant_metric_cols'].mean():.1f}"
)
re2.groupby("fault")[["metrics_cols", "logs_rows", "traces_rows", "constant_metric_cols"]].mean().round(2)


RE2-OB cases: 90
Fault counts:
 fault
cpu       15
delay     15
disk      15
loss      15
mem       15
socket    15
Name: count, dtype: int64
Service counts:
 service
checkoutservice          18
currencyservice          18
emailservice             18
productcatalogservice    18
recommendationservice    18
Name: count, dtype: int64
Telemetry averages — metrics_cols: 73.4, logs: 167258, traces: 382903, missing: 0.0039, constant cols: 15.1


## 7. Cleaning rules from EDA

1. Drop constant / near-constant metric columns before ranking.  
2. Align analysis to `inject_time`; use pre-fault window as baseline.  
3. Never put case folder names into model prompts (**label leakage**).  
4. Map metric columns → services via suffixes (`_cpu`, `_mem`, `_latency-90`, …).  
5. Treat `redis_*` metrics as `redis-cart` when matching Online Boutique graph names.  
6. RE1 is metrics-only; do not expect logs/traces there.  
7. Choose a post-fault **time window** carefully (RE2-OB sweep preferred **120s** for our simple ranker features).


## 8. Link to scoring (not trained in this notebook)

After EDA we score methods with **AC@1 / AC@3 / Avg@5** (is the true service ranked #1 / in top-3 / average over top-5).  
Existing baselines (BARO, N-Sigma, E-Diagnosis) and a small GBT ranker live under `eval/rcaeval/` on the sprint branch.  
This notebook is the **data-understanding** deliverable only.


## Reproduce

```bash
# from repo root
uv sync --dev
uv run jupyter notebook EDA/RCAEval_EDA.ipynb
# or
uv run python -c "print('open EDA/RCAEval_EDA.ipynb')"
```

Data path: set `RCAEVAL_DIR` or use `../datasets/RCAEval`.
